In [108]:
!pip install -U langgraph langchain langchain-openai

The API Key should be named "OPENAI_API_KEY", while creating the key (https://openrouter.ai/settings/keys)

In [109]:
import os
os.environ["OPENAI_API_KEY"] = ""
# Enter your own API KEY

In [110]:
import os
from dotenv import load_dotenv

load_dotenv()

False

In [111]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

In [112]:
from typing_extensions import TypedDict

class CounterState(TypedDict):
    count: int

In [113]:
def increment(state: CounterState) -> dict:
    state["count"] += 1
    return state

In [114]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(CounterState)

builder.add_node("increment", increment)

# Define the execution order: START -> increment -> END
builder.add_edge(START, "increment")
builder.add_edge("increment", END)

graph = builder.compile()

In [115]:
initial_state: CounterState = {"count": 0}

result = graph.invoke(initial_state)

print(result)

# Output
# {'count': 1}

{'count': 1}


In [116]:
from typing import Literal

def should_continue(state: CounterState) -> Literal["increment", END]:
    print("count is", state["count"])

    if state["count"] <= 15: # keep looping
        return "increment"

    return END # stop the graph

In [117]:
builder = StateGraph(CounterState)

builder.add_node("increment", increment)

builder.add_edge(START, "increment")

builder.add_conditional_edges(
    "increment",
    should_continue,
    ["increment", END],
)

graph = builder.compile()

result = graph.invoke({"count": 0})

print(result)

# Output
# {'count': 3}

count is 1
count is 2
count is 3
count is 4
count is 5
count is 6
count is 7
count is 8
count is 9
count is 10
count is 11
count is 12
count is 13
count is 14
count is 15
count is 16
{'count': 16}


In [118]:
from typing_extensions import TypedDict

class AgentState(TypedDict):
    user_input: str
    response: str

In [119]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

In [120]:
from langchain_core.messages import SystemMessage, HumanMessage

def llm_node(state: AgentState) -> dict:
    messages = [
        SystemMessage(content="You are a helpful assistant."),
        HumanMessage(content=state["user_input"]),
    ]

    reply = llm.invoke(messages)

    return {"response": reply.content}

In [121]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(AgentState)

builder.add_node("llm", llm_node)

# define the flow: START -> llm -> END
builder.add_edge(START, "llm")
builder.add_edge("llm", END)

graph = builder.compile()

In [122]:
initial_state: AgentState = {
    "user_input": "Tell me about Deven Mukai from Creighton University",
    "response": "",
}

result = graph.invoke(initial_state)

print("User:", initial_state["user_input"])
print("Assistant:", result["response"])

User: Tell me about Deven Mukai from Creighton University
Assistant: As of my last knowledge update in October 2021, I don't have specific information regarding an individual named Deven Mukai associated with Creighton University. It's possible that he may have become notable or gained recognition after that time, or he may not be widely known outside of particular academic or community circles. 

If you have specific details or context about Deven Mukai, such as his field of study, accomplishments, or contributions, I would be happy to help you explore that more. Otherwise, I recommend checking the Creighton University website or other reputable sources to find up-to-date information.


In [123]:
state2: AgentState = {
    "user_input": "Can you summarise that in two lines?",
    "response": "",
}
result2 = graph.invoke(state2)

print("\nTurn 2 - User:", state2["user_input"])
print("Turn 2 - Assistant:", result2["response"])


Turn 2 - User: Can you summarise that in two lines?
Turn 2 - Assistant: Sure! Please provide the text or information you would like summarized in two lines.


In [124]:
from typing_extensions import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    foo: str
    bar: Annotated[list[str], add]

def node_a(state: State):
    # overwrite foo, append "a" to bar
    return {"foo": "a", "bar": ["a"]}

def node_b(state: State):
    # overwrite foo, append "b" to bar
    return {"foo": "b", "bar": ["b"]}

In [125]:
builder = StateGraph(State)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)

builder.add_edge(START, "node_a")
builder.add_edge("node_a", "node_b")
builder.add_edge("node_b", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [126]:
config = {"configurable": {"thread_id": "1"}}

final_state = graph.invoke({"foo": "", "bar": []}, config=config)
print("Final state:", final_state)

# Output
# Final state: {'foo': 'b', 'bar': ['a', 'b']}

Final state: {'foo': 'b', 'bar': ['a', 'b']}


In [127]:
history = list(graph.get_state_history(config))

for i, snap in enumerate(history[::-1]):
    print(f"\nCheckpoint {i}:")
    print("  created_at:", snap.created_at)
    print("  node:", snap.metadata)
    print("  values:", snap.values)


Checkpoint 0:
  created_at: 2025-11-20T22:21:59.661752+00:00
  node: {'source': 'input', 'step': -1, 'parents': {}}
  values: {'bar': []}

Checkpoint 1:
  created_at: 2025-11-20T22:21:59.666547+00:00
  node: {'source': 'loop', 'step': 0, 'parents': {}}
  values: {'foo': '', 'bar': []}

Checkpoint 2:
  created_at: 2025-11-20T22:21:59.673356+00:00
  node: {'source': 'loop', 'step': 1, 'parents': {}}
  values: {'foo': 'a', 'bar': ['a']}

Checkpoint 3:
  created_at: 2025-11-20T22:21:59.676883+00:00
  node: {'source': 'loop', 'step': 2, 'parents': {}}
  values: {'foo': 'b', 'bar': ['a', 'b']}


In [128]:
from typing_extensions import TypedDict, Annotated
from langchain_core.messages import AnyMessage
import operator

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [129]:
from langchain_core.messages import SystemMessage

def chat_llm_node(state: MessagesState) -> dict:

    # build prompt from system + existing conversation
    history = [SystemMessage(content="You are a helpful assistant.")]
    history.extend(state["messages"])

    reply = llm.invoke(history)

    return {"messages": [reply]}

In [130]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage

checkpointer = InMemorySaver()

builder = StateGraph(MessagesState)
builder.add_node("chat_llm", chat_llm_node)
builder.add_edge(START, "chat_llm")
builder.add_edge("chat_llm", END)

graph = builder.compile(checkpointer=checkpointer)

In [131]:
config = {"configurable": {"thread_id": "user_123"}}


In [132]:
config = {"configurable": {"thread_id": "user_123"}}

# Turn 1
state1 = {"messages": [HumanMessage(content="What is GIL in python?")]}
result1 = graph.invoke(state1, config=config)

for m in result1["messages"]:
    print(type(m).__name__, ":", m.content)

# Turn 2
state2 = {"messages": [HumanMessage(content="Summarise it in two lines")],}
result2 = graph.invoke(state2, config=config)

for m in result2["messages"][-2:]:
    print(type(m).__name__, ":", m.content)

HumanMessage : What is GIL in python?
AIMessage : The Global Interpreter Lock (GIL) is a mechanism used in Python, specifically in the CPython implementation (the standard and most commonly used implementation of Python), to manage access to Python objects and facilitate memory management among multiple threads. The GIL prevents multiple native threads from executing Python bytecodes at once, thereby making Python inherently not fully multithreaded.

### Key Characteristics of the GIL:

1. **Thread Safety**: The GIL ensures that only one thread can execute Python bytecode at a time, simplifying memory management and preventing race conditions in memory allocations.

2. **Performance Trade-offs**: While the GIL makes it easier to work with Python's memory management, it can lead to performance bottlenecks in CPU-bound programs that attempt to leverage multiple threads for parallel execution. In I/O-bound applications (e.g., when waiting for network responses or reading from files), thre